# Exploratory Data Analysis: Sri Lankan A/L University Cutoffs
This notebook explores the dataset extracted from the official UGC (University Grants Commission) Sri Lanka cutoff publications. It covers:
1. Data Loading & Preprocessing
2. General Distributions (Z-Scores, Intake)
3. Course & University Competitiveness
4. District-wise Disparities
5. Year-over-Year (YoY) Trends


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 6)


## 1. Data Loading
We will load the denormalized data directly from our SQLite database.

In [ ]:
# Connect to the SQLite database
db_path = '../data/bronze/db/al_cutoffs.db'
conn = sqlite3.connect(db_path)

# Load the main denormalized dataset
query_cutoffs = '''
SELECT 
    f.AcademicYear, f.ExamYear, c.CourseName, u.UniversityName, d.DistrictName, 
    f.CutoffZ, f.CutoffStatus, f.AptitudeTest
FROM fact_cutoffs f
JOIN dim_course c ON f.CourseID = c.CourseID
JOIN dim_university u ON f.UniversityID = u.UniversityID
JOIN dim_district d ON f.DistrictID = d.DistrictID
WHERE f.CutoffZ IS NOT NULL
'''
df_cutoffs = pd.read_sql(query_cutoffs, conn)

# Load Intake Data
query_intake = '''
SELECT i.AcademicYear, c.CourseName, i.Intake
FROM fact_course_intake i
JOIN dim_course c ON i.CourseID = c.CourseID
'''
df_intake = pd.read_sql(query_intake, conn)

conn.close()

# Preview Data
display(df_cutoffs.head())
print(f"Total cutoff records: {len(df_cutoffs):,}")


## 2. General Distributions

In [ ]:
# Z-Score Overall Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_cutoffs['CutoffZ'], bins=50, kde=True)
plt.title('Distribution of Z-Scores Across All Years, Courses, and Districts')
plt.xlabel('Z-Score')
plt.ylabel('Frequency')
plt.show()

print(df_cutoffs['CutoffZ'].describe())


In [ ]:
# Total Intake Over Time
intake_by_year = df_intake.groupby('AcademicYear')['Intake'].sum().reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=intake_by_year, x='AcademicYear', y='Intake', color='steelblue')
plt.title('Total Planned University Intake by Academic Year')
plt.xlabel('Academic Year')
plt.ylabel('Total Seats')
plt.show()


## 3. Course & University Competitiveness
Which courses and universities require the highest Z-scores?

In [ ]:
# Top 15 Most Competitive Courses (by Median Z-Score)
top_courses = df_cutoffs.groupby('CourseName')['CutoffZ'].median().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_courses.values, y=top_courses.index, palette='Reds_r')
plt.title('Top 15 Most Competitive Courses (Median Z-Score)')
plt.xlabel('Median Z-Score')
plt.ylabel('')
plt.show()


In [ ]:
# Top 15 Most Competitive Universities (by Median Z-Score)
top_unis = df_cutoffs.groupby('UniversityName')['CutoffZ'].median().sort_values(ascending=False).head(15)

plt.figure(figsize=(12, 6))
sns.barplot(x=top_unis.values, y=top_unis.index, palette='Blues_r')
plt.title('Top 15 Most Competitive Universities (Median Z-Score)')
plt.xlabel('Median Z-Score')
plt.ylabel('')
plt.show()


## 4. District-wise Disparities
In Sri Lanka, district quotas affect cutoffs heavily. Let's look at the Medicine course as a baseline.

In [ ]:
# Filter for Medicine
df_med = df_cutoffs[df_cutoffs['CourseName'] == 'MEDICINE']

# Order districts by their median Medicine cutoff
district_order = df_med.groupby('DistrictName')['CutoffZ'].median().sort_values(ascending=False).index

plt.figure(figsize=(14, 8))
sns.boxplot(data=df_med, x='CutoffZ', y='DistrictName', order=district_order, palette='viridis')
plt.title('Z-Score Disparity for MEDICINE by District (All Years)')
plt.xlabel('Z-Score Cutoff')
plt.ylabel('District')
plt.show()


## 5. Year-over-Year (YoY) Trends
How are the cutoffs changing over time for highly sought-after programs?

In [ ]:
# Trend for Medicine at Colombo vs Peradeniya vs Moratuwa (for Colombo District students)
df_trend = df_cutoffs[
    (df_cutoffs['CourseName'] == 'MEDICINE') & 
    (df_cutoffs['DistrictName'] == 'COLOMBO') &
    (df_cutoffs['UniversityName'].isin(['University of Colombo', 'University of Peradeniya', 'University of Kelaniya']))
]

plt.figure(figsize=(10, 6))
sns.lineplot(data=df_trend, x='AcademicYear', y='CutoffZ', hue='UniversityName', marker='o', linewidth=2)
plt.title('YoY Medicine Z-Score Trend for Colombo District Students')
plt.xlabel('Academic Year')
plt.ylabel('Z-Score Cutoff')
plt.legend(title='University')
plt.grid(True)
plt.show()


In [ ]:
# Engineering vs Medicine (National Average Trend)
df_eng_med = df_cutoffs[df_cutoffs['CourseName'].isin(['MEDICINE', 'ENGINEERING'])]

plt.figure(figsize=(10, 6))
sns.lineplot(data=df_eng_med, x='AcademicYear', y='CutoffZ', hue='CourseName', marker='o', estimator='median', errorbar=None, linewidth=2)
plt.title('Median Z-Score Trend: Engineering vs Medicine')
plt.xlabel('Academic Year')
plt.ylabel('Median Z-Score (Across all Districts & Unis)')
plt.show()
